# Current noise from counting statistics

This tutorial was written by **Simon Wozny** and adapted from his [QmeQ noise example](https://github.com/si8881wo/qmeq-noise-example) at [source commit 0f81175](https://github.com/si8881wo/qmeq-noise-example/commit/0f81175c63b4f3846ac9c392c739615b038b9054) for inclusion in QmeQ. The original is copyright 2024 Simon Wozny and distributed under the BSD 2-Clause License; the retained license is in `examples/licenses/qmeq-noise-example-BSD-2-Clause.txt`.

We compare Lindblad counting at the left leads with fourth-order RTD counting at the right leads for a spinful Anderson dot.

In [ ]:
import qmeq
import numpy as np
import matplotlib.pyplot as plt

## Spinful single Anderson dot

The dot has two spin levels, a Coulomb interaction, and four lead channels: left and right for each spin. The two sides have different temperatures, so the setup produces thermoelectric current even at zero voltage bias.

> **RTD limitation:** the unequal-temperature RTD integral path is still being validated for residual finite-bandwidth dependence. The RTD thermal-bias curve below demonstrates the counting API; do not use it as a reference-precision result without an independent bandwidth-convergence check.

In [ ]:
n = 2
h = {(0, 0): 500, (1, 1): 500}
U = {(0, 1, 1, 0): 2000}

T = 100
nleads = 4
mulst = {0: 0, 1: 0, 2: 0, 3: 0}
tlst = {0: T, 1: 2*T, 2: T, 3: 2*T}

gammaL = 1.5
gammaR = 0.5
tL = np.sqrt(gammaL / (2*np.pi))
tR = np.sqrt(gammaR / (2*np.pi))
tleads = {(0, 0): tL, (1, 0): tR, (2, 1): tL, (3, 1): tR}

## Select the counted leads

One counting field aggregates the selected leads. We count the two left spin channels with Lindblad and the two right spin channels with RTD. RTD counting currently requires `off_diag_corrections=False`.

In [ ]:
common = dict(nsingle=n, hsingle=h, coulomb=U, nleads=nleads,
              mulst=mulst, tlst=tlst, tleads=tleads, dband=1e4)

system_L = qmeq.Builder(
    **common, countingleads=[0, 2], kerntype='pyLindblad'
)
system_R = qmeq.Builder(
    **common, countingleads=[1, 3], kerntype='pyRTDnoise',
    off_diag_corrections=False,
)

## Current and zero-frequency noise

`system.current` remains lead resolved. `system.current_noise` contains `[I, S]` for the aggregate counted leads. QmeQ defines positive current as flowing from a lead into the dot, so left and right currents have opposite signs in a stationary two-terminal setup.

In [ ]:
system_L.solve()
system_R.solve()

print('Lead-resolved current, Lindblad:', system_L.current)
print('Lead-resolved current, RTD:', system_R.current)
print('Left aggregate [I, S], Lindblad:', system_L.current_noise)
print('Right aggregate [I, S], RTD:', system_R.current_noise.real)

## Gate sweep

A normal parameter sweep needs no special counting-statistics machinery: change the Hamiltonian, solve again, and record the two cumulants.

In [ ]:
Vg = np.linspace(-1500, 3500, 100)
I = [[], []]
I_noise = [[], []]
S = [[], []]

for i, system in enumerate([system_L, system_R]):
    for vg in Vg:
        system.change(hsingle={(0, 0): -vg, (1, 1): -vg})
        system.solve()
        counted = (0, 2) if system is system_L else (1, 3)
        I[i].append(sum(system.current[lead] for lead in counted))
        I_noise[i].append(system.current_noise[0].real)
        S[i].append(system.current_noise[1].real)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(Vg, I[0], label='left (ordinary)', lw=3)
ax[0].plot(Vg, I[1], label='right (ordinary)', lw=3)
ax[0].plot(Vg, I_noise[0], label='left (counting)')
ax[0].plot(Vg, I_noise[1], label='right (counting)')
ax[0].set(xlabel=r'$V_g$', ylabel=r'$I$')
ax[0].legend()

ax[1].plot(Vg, S[0], label='left')
ax[1].plot(Vg, S[1], label='right')
ax[1].set(xlabel=r'$V_g$', ylabel=r'$S$')
ax[1].legend()
fig.tight_layout()